In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
DATA = Path("whatsapp-data")
daily  = pd.read_csv(DATA/"wa_daily.csv",parse_dates=["date"])
hourly = pd.read_csv(DATA/"wa_hourly.csv",parse_dates=["date"])
chats  = pd.read_csv(DATA/"wa_chat_summary.csv",parse_dates=["first_date","last_date"])
msgs   = pd.read_csv(DATA/"wa_messages.csv",parse_dates=["timestamp", "date"])

In [3]:
for ad, df in [("daily", daily), ("hourly", hourly), ("chats", chats), ("msgs", msgs)]:
    print(ad, df.shape)
    display(df.head(3))

daily (842, 9)


,date,self_msg_count,self_char_count,self_active_chat_count,other_msg_count,other_char_count,self_night_ratio,self_first_msg_hour,self_last_msg_hour
0,2024-08-05,0,0,0,0,0,NaN,NaN,NaN
1,2024-08-06,31,641,3,80,1427,0.0,9.42,22.45
2,2024-08-07,29,920,1,29,967,0.0,10.58,19.25


hourly (20208, 6)


,date,hour,self_msg_count,self_char_count,other_msg_count,other_char_count
0,2024-08-05,0,0,0,0,0
1,2024-08-05,1,0,0,0,0
2,2024-08-05,2,0,0,0,0


chats (114, 15)


,chat_id,chat_type,num_participants,first_date,last_date,active_days_count,self_msg_count,other_msg_count,self_char_count,other_char_count,self_msg_share,session_count,self_session_starter_share,median_response_delay_self_sec,median_response_delay_other_sec
0,c001,direct,2,2025-05-12,2026-03-02,4,7,8,233,720,0.4667,4,0.25,120.0,60.0
1,c002,direct,2,2025-12-09,2026-11-23,3,33,12,355,319,0.7333,3,0.00,1020.0,90.0
2,c003,direct,2,2025-03-07,2025-05-30,5,35,30,1170,560,0.5385,5,0.40,60.0,60.0


msgs (43474, 32)


,msg_id,chat_id,chat_type,num_participants,sender_role,sender_id,timestamp,date,hour,minute,...,is_media,is_deleted,is_forwarded,is_paste_suspected,prev_sender_role,time_since_prev_msg_sec,session_id,is_session_starter,turn_id,response_delay_sec
0,1,c001,direct,2,other,u01,2025-05-12 23:02:00,2025-05-12,23,2,...,0,0,0,1,NaN,NaN,1,1,1,NaN
1,2,c001,direct,2,other,u01,2025-05-12 23:03:00,2025-05-12,23,3,...,0,0,0,0,other,60.0,1,0,1,NaN
2,3,c001,direct,2,other,u01,2025-05-12 23:04:00,2025-05-12,23,4,...,0,0,0,0,other,60.0,1,0,1,NaN


In [4]:
daily["weekday"]=daily["date"].dt.dayofweek  
daily["month"]=daily["date"].dt.month
daily["is_weekend"]=daily["weekday"]>=5
hourly["weekday"]=hourly["date"].dt.dayofweek
hourly["is_weekend"]=hourly["weekday"] >= 5

In [5]:
for ad,df in [("daily", daily),("hourly", hourly),("chats", chats),("msgs", msgs)]:
    print(ad)
    print(df.isna().sum()[df.isna().sum()>0])

daily
self_night_ratio       77
self_first_msg_hour    77
self_last_msg_hour     77
dtype: int64
hourly
Series([], dtype: int64)
chats
median_response_delay_self_sec     3
median_response_delay_other_sec    2
dtype: int64
msgs
prev_sender_role             114
time_since_prev_msg_sec      114
response_delay_sec         26735
dtype: int64


In [6]:
# Flag columns: binary (0/1) indicators describing special message types.
# Averaging a 0/1 column gives the share of messages with that flag.
flag_cols = ["is_media", "is_deleted", "is_forwarded", "is_paste_suspected", "has_url"]
# Share of each flag, in percent.
# Why we check this: we need to know how much of the data will be excluded
# from text-based analyses (message length, emoji use, etc.).
msgs[flag_cols].mean().round(4) * 100
# Raw counts of each flag.
# is_media / is_deleted: these rows have no real text (e.g. "<Media omitted>"),
#   so their char_count is meaningless and would distort length statistics.
# is_forwarded: forwarded messages are not written by the sender, so they do not
#   reflect the sender's own style. (If this is always 0, the export simply has
#   no forwarding info.)
# is_paste_suspected: very long pasted text (e.g. notes, copied messages) is an
#   outlier in typing behavior and would inflate mean message length.
# has_url: links are kept, but their length is driven by the URL, not by writing style.
msgs[flag_cols].sum()

is_media              3822
is_deleted              99
is_forwarded             0
is_paste_suspected    7602
has_url                805
dtype: int64

In [7]:
# 1) Is the flag just overlapping with media / URL rows?
pd.crosstab(msgs["is_paste_suspected"], [msgs["is_media"], msgs["has_url"]])
# 2) What does a flagged message look like (lengths, word counts)?
msgs.groupby("is_paste_suspected")[["char_count", "word_count", "line_count"]].describe().T
# 3) Share of flagged rows among text-only messages
text_only = msgs[(msgs["is_media"] == 0) & (msgs["is_deleted"] == 0)]
text_only["is_paste_suspected"].mean()

0.19219781053270296

In [8]:
text_only = msgs[(msgs["is_media"] == 0) & (msgs["is_deleted"] == 0)]
# What do flagged vs. non-flagged messages look like?
text_only.groupby("is_paste_suspected")[["char_count", "word_count", "line_count"]].describe().T

is_paste_suspected             0             1
char_count count    31951.000000   7602.000000
           mean        26.012425     51.513944
           std         21.094812    584.923594
           min          0.000000      1.000000
           25%         14.000000      3.000000
           50%         21.000000      8.000000
           75%         32.000000     13.000000
           max        490.000000  27340.000000
word_count count    31951.000000   7602.000000
           mean         4.333667      6.045251
           std          3.234196     71.234784
           min          0.000000      1.000000
           25%          2.000000      1.000000
           50%          4.000000      1.000000
           75%          5.000000      2.000000
           max         63.000000   3304.000000
line_count count    31951.000000   7602.000000
           mean         0.990204      1.484478
           std          0.149484      9.766600
           min          0.000000      1.000000
           25%          1.000000      1.000000
           50%          1.000000      1.000000
           75%          1.000000      1.000000
           max          4.000000    535.000000

In [9]:
# How many messages are truly extreme? Check the tail instead of trusting the flag.
text_only = msgs[(msgs["is_media"] == 0) & (msgs["is_deleted"] == 0)]
text_only["char_count"].quantile([.99, .999, .9999])
(text_only["char_count"] > 500).sum()

90

In [10]:
# Main subset: drop media and deleted rows (no real text).
msgs_clean = msgs[(msgs["is_media"] == 0) & (msgs["is_deleted"] == 0)].copy()
# Own outlier rule instead of is_paste_suspected: very long messages (> 500 chars).
msgs_clean["is_long_outlier"] = msgs_clean["char_count"] > 500
# Sensitivity subset: same data without the long outliers.
msgs_trim = msgs_clean[~msgs_clean["is_long_outlier"]].copy()
print(len(msgs), len(msgs_clean), len(msgs_trim))

43474 39553 39463
